# ViT5 Fine-tuning — MCQ Generation (Stage A: QG + Stage B: DG)

**Yêu cầu:** Runtime → Change runtime type → **A100 GPU**

**Luồng:**
1. Mount Google Drive
2. Cài thư viện
3. Load `qa_pairs.parquet` + `corpus.parquet` từ Drive
4. Chuẩn bị dataset cho Stage A (QG) và Stage B (DG)
5. Train Stage A → lưu model
6. Train Stage B → lưu model
7. Inference: sinh câu hỏi + distractor → `vit5_ft.jsonl`
8. Copy kết quả về Drive

## 0. Kiểm tra GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
print('CUDA:', torch.version.cuda)

## 1. Cài thư viện

In [ ]:
%%capture
!pip install transformers==4.40.0 datasets accelerate sentencepiece pydantic pandas pyarrow

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ==============================================================
# SỬA đường dẫn này cho khớp với vị trí project của bạn trên Drive
PROJECT_DIR = '/content/drive/MyDrive/files_v1'
# ==============================================================

import os
os.makedirs(f'{PROJECT_DIR}/data/generated', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models/vit5_qg', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models/vit5_dg', exist_ok=True)
print('Project dir:', PROJECT_DIR)

## 3. Load dữ liệu

In [ ]:
import pandas as pd

qa  = pd.read_parquet(f'{PROJECT_DIR}/data/processed/qa_pairs.parquet')
ctx = pd.read_parquet(f'{PROJECT_DIR}/data/processed/corpus.parquet')

print(f'QA pairs : {len(qa):,}')
print(f'Corpus   : {len(ctx):,} đoạn')
print(f'Splits QA: {qa.split.value_counts().to_dict()}')

# Chỉ lấy câu có span hợp lệ + answerable
qa_clean = qa[(qa.answer_span_ok == True) & (qa.is_impossible == False)].copy()
print(f'\nSau lọc (span OK + answerable): {len(qa_clean):,}')

# Merge với corpus để lấy context đầy đủ
df = qa_clean.merge(ctx[['context_id','context','title','is_vietnam']],
                    on='context_id', how='inner')
print(f'Sau merge với corpus: {len(df):,}')

## 4. Suy ra wh_type từ câu hỏi + answer
Dùng để tạo trường `<type>` trong input của Stage A.

In [ ]:
import re

YEAR_RE = re.compile(r'^\d{3,4}$')
DATE_RE = re.compile(r'\d{1,2}/\d{1,2}/\d{4}')

WH_KEYWORDS = {
    'thoi_gian':   ['khi nào','năm nào','bao giờ','thời gian','thế kỷ','ngày nào','tháng'],
    'nhan_vat':    ['ai ','người nào','nhân vật','ông nào','bà nào','vua nào','tướng nào'],
    'dia_diem':    ['ở đâu','tại đâu','địa điểm','nơi nào','thành phố','quốc gia','vùng'],
    'su_kien':     ['sự kiện nào','điều gì','chuyện gì','việc gì'],
    'nguyen_nhan': ['vì sao','tại sao','do đâu','nguyên nhân','lý do'],
    'y_nghia':     ['ý nghĩa','kết quả','hệ quả','tác động','dẫn đến','tầm quan trọng'],
}

def infer_wh_type(question: str, answer: str) -> str:
    # answer type ưu tiên hơn từ hỏi
    ans = answer.strip()
    if YEAR_RE.match(ans) or DATE_RE.search(ans):
        return 'thoi_gian'
    q = question.lower()
    for wh, keywords in WH_KEYWORDS.items():
        if any(k in q for k in keywords):
            return wh
    return 'su_kien'  # fallback

df['wh_type'] = df.apply(
    lambda r: infer_wh_type(r['question'], r['answer_text']), axis=1)
print('Phân bố wh_type:')
print(df.wh_type.value_counts())

## 5. Tạo dataset Stage A (Question Generation)

```
Input:  '<type> thoi_gian </type> <ans> 1954 </ans> <ctx> Chiến thắng... </ctx>'
Target: 'Chiến thắng Điện Biên Phủ diễn ra vào năm nào?'
```

In [ ]:
MAX_SRC_LEN = 384
MAX_TGT_LEN = 80

def make_qg_input(row):
    return (f"<type> {row['wh_type']} </type> "
            f"<ans> {row['answer_text']} </ans> "
            f"<ctx> {row['context'][:800]} </ctx>")

df['qg_input']  = df.apply(make_qg_input, axis=1)
df['qg_target'] = df['question']

# chia theo split
train_df = df[df['split'] == 'train'][['qg_input','qg_target']].reset_index(drop=True)
dev_df   = df[df['split'] == 'dev' ][['qg_input','qg_target']].reset_index(drop=True)

print(f'Stage A — Train: {len(train_df):,}  |  Dev: {len(dev_df):,}')
print('\nVí dụ input:')
print(train_df.qg_input.iloc[0][:200])
print('\nVí dụ target:')
print(train_df.qg_target.iloc[0])

## 6. Tạo dataset Stage B (Distractor Generation)

```
Input:  '<q> Chiến thắng năm nào? </q> <ans> 1954 </ans> <ctx> ... </ctx>'
Target: '1945 | 1975 | 1986'
```

Distractor gold lấy từ `plausible_answer` của ViQuAD (is_impossible=True cùng context)
và từ các answer khác trong cùng `context_id`.

In [ ]:
# Tạo distractor gold: lấy các answer khác trong cùng context
from collections import defaultdict

ctx_answers = defaultdict(set)
for _, row in df.iterrows():
    ctx_answers[row['context_id']].add(row['answer_text'])

# Thêm plausible_answer từ câu is_impossible
plausible = qa[qa.is_impossible == True][['context_id','plausible_answer']]
for _, row in plausible.iterrows():
    if row['plausible_answer']:
        ctx_answers[row['context_id']].add(str(row['plausible_answer']))

def make_dg_sample(row):
    """Tạo mẫu DG: lấy tối đa 3 answer khác trong cùng context làm distractor gold."""
    ans = row['answer_text']
    pool = [a for a in ctx_answers[row['context_id']]
            if a != ans and a.strip()]
    if len(pool) < 1:
        return None
    distractors = pool[:3]
    return {
        'dg_input':  (f"<q> {row['question']} </q> "
                      f"<ans> {ans} </ans> "
                      f"<ctx> {row['context'][:600]} </ctx>"),
        'dg_target': ' | '.join(distractors),
    }

dg_rows = []
for _, row in df.iterrows():
    r = make_dg_sample(row)
    if r:
        r['split'] = row['split']
        dg_rows.append(r)

dg_df = pd.DataFrame(dg_rows)
train_dg = dg_df[dg_df['split'] == 'train'][['dg_input','dg_target']].reset_index(drop=True)
dev_dg   = dg_df[dg_df['split'] == 'dev' ][['dg_input','dg_target']].reset_index(drop=True)

print(f'Stage B — Train: {len(train_dg):,}  |  Dev: {len(dev_dg):,}')
print('\nVí dụ input DG:')
print(train_dg.dg_input.iloc[0][:200])
print('\nVí dụ target DG:')
print(train_dg.dg_target.iloc[0])

## 7. Load tokenizer + model ViT5-base

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = 'VietAI/vit5-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded:', MODEL_NAME)
print('Vocab size:', tokenizer.vocab_size)

## 8. Hàm tokenize chung

In [ ]:
from datasets import Dataset

def tokenize_fn(examples, src_col, tgt_col):
    model_inputs = tokenizer(
        examples[src_col],
        max_length=MAX_SRC_LEN,
        truncation=True,
        padding='max_length',
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples[tgt_col],
            max_length=MAX_TGT_LEN,
            truncation=True,
            padding='max_length',
        )
    # -100 để loss bỏ qua padding
    labels['input_ids'] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in lbl]
        for lbl in labels['input_ids']
    ]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print('Tokenize function defined.')

## 9. Train Stage A — Question Generation
~1.5–2 giờ trên A100

In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM, Seq2SeqTrainer,
    Seq2SeqTrainingArguments, DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
import numpy as np

# --- Dataset ---
ds_train_a = Dataset.from_pandas(train_df)
ds_dev_a   = Dataset.from_pandas(dev_df)

tok_train_a = ds_train_a.map(
    lambda x: tokenize_fn(x, 'qg_input', 'qg_target'),
    batched=True, remove_columns=ds_train_a.column_names)
tok_dev_a = ds_dev_a.map(
    lambda x: tokenize_fn(x, 'qg_input', 'qg_target'),
    batched=True, remove_columns=ds_dev_a.column_names)

# --- Model ---
model_a = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model_a = model_a.cuda()

# --- Training args ---
QG_OUTPUT = f'{PROJECT_DIR}/models/vit5_qg'

args_a = Seq2SeqTrainingArguments(
    output_dir=QG_OUTPUT,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,   # effective batch = 32
    learning_rate=3e-5,
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=True,                        # A100 hỗ trợ tốt
    predict_with_generate=True,
    generation_max_length=MAX_TGT_LEN,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=100,
    report_to='none',
    seed=42,
)

collator_a = DataCollatorForSeq2Seq(tokenizer, model=model_a, padding=True)

trainer_a = Seq2SeqTrainer(
    model=model_a,
    args=args_a,
    train_dataset=tok_train_a,
    eval_dataset=tok_dev_a,
    tokenizer=tokenizer,
    data_collator=collator_a,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Bắt đầu train Stage A (QG)...')
print(f'Train: {len(tok_train_a):,} | Dev: {len(tok_dev_a):,}')
trainer_a.train()
trainer_a.save_model(QG_OUTPUT)
tokenizer.save_pretrained(QG_OUTPUT)
print(f'\n✅ Stage A xong → {QG_OUTPUT}')

## 10. Train Stage B — Distractor Generation
~1–1.5 giờ trên A100

In [ ]:
# Giải phóng VRAM trước khi train Stage B
del model_a
torch.cuda.empty_cache()
print('VRAM freed.')

# --- Dataset ---
ds_train_b = Dataset.from_pandas(train_dg)
ds_dev_b   = Dataset.from_pandas(dev_dg)

tok_train_b = ds_train_b.map(
    lambda x: tokenize_fn(x, 'dg_input', 'dg_target'),
    batched=True, remove_columns=ds_train_b.column_names)
tok_dev_b = ds_dev_b.map(
    lambda x: tokenize_fn(x, 'dg_input', 'dg_target'),
    batched=True, remove_columns=ds_dev_b.column_names)

# --- Model (khởi tạo lại từ pretrained) ---
model_b = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model_b = model_b.cuda()

DG_OUTPUT = f'{PROJECT_DIR}/models/vit5_dg'

args_b = Seq2SeqTrainingArguments(
    output_dir=DG_OUTPUT,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=60,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=100,
    report_to='none',
    seed=42,
)

collator_b = DataCollatorForSeq2Seq(tokenizer, model=model_b, padding=True)

trainer_b = Seq2SeqTrainer(
    model=model_b,
    args=args_b,
    train_dataset=tok_train_b,
    eval_dataset=tok_dev_b,
    tokenizer=tokenizer,
    data_collator=collator_b,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Bắt đầu train Stage B (DG)...')
print(f'Train: {len(tok_train_b):,} | Dev: {len(tok_dev_b):,}')
trainer_b.train()
trainer_b.save_model(DG_OUTPUT)
tokenizer.save_pretrained(DG_OUTPUT)
print(f'\n✅ Stage B xong → {DG_OUTPUT}')

## 11. Inference — sinh câu hỏi trên test set
Dùng model Stage A sinh câu hỏi, Stage B sinh distractor.
Output: `vit5_ft.jsonl` cùng format với rule_based và rag_llm.

In [ ]:
import json, sys, hashlib, random
from pathlib import Path

# Load 2 model đã train
del model_b
torch.cuda.empty_cache()

model_qg = AutoModelForSeq2SeqLM.from_pretrained(QG_OUTPUT).cuda().eval()
model_dg = AutoModelForSeq2SeqLM.from_pretrained(DG_OUTPUT).cuda().eval()
tok_inf  = AutoTokenizer.from_pretrained(QG_OUTPUT)
print('Models loaded.')

In [ ]:
def generate_question(context: str, answer: str, wh_type: str) -> str:
    src = (f"<type> {wh_type} </type> "
           f"<ans> {answer} </ans> "
           f"<ctx> {context[:800]} </ctx>")
    ids = tok_inf(src, return_tensors='pt',
                  max_length=MAX_SRC_LEN, truncation=True).input_ids.cuda()
    with torch.no_grad():
        out = model_qg.generate(
            ids, max_new_tokens=MAX_TGT_LEN,
            num_beams=4, no_repeat_ngram_size=3,
            early_stopping=True,
        )
    return tok_inf.decode(out[0], skip_special_tokens=True)


def generate_distractors(question: str, answer: str, context: str) -> list:
    src = (f"<q> {question} </q> "
           f"<ans> {answer} </ans> "
           f"<ctx> {context[:600]} </ctx>")
    ids = tok_inf(src, return_tensors='pt',
                  max_length=MAX_SRC_LEN, truncation=True).input_ids.cuda()
    with torch.no_grad():
        out = model_dg.generate(
            ids, max_new_tokens=60,
            num_beams=4, no_repeat_ngram_size=2,
            early_stopping=True,
        )
    raw = tok_inf.decode(out[0], skip_special_tokens=True)
    parts = [d.strip() for d in raw.split('|') if d.strip()]
    return parts[:3]


def finalize_options(correct, distractors, seed=None):
    """Khử trùng, đủ 4 phương án, xáo trộn A/B/C/D."""
    seen = {correct.lower()}
    clean = []
    for d in distractors:
        if d.lower() not in seen and d.strip():
            seen.add(d.lower()); clean.append(d)
    if len(clean) < 3:
        return None, None   # không đủ distractor
    entries = [(correct, True)] + [(d, False) for d in clean[:3]]
    rng = random.Random(seed)
    rng.shuffle(entries)
    labels = ['A','B','C','D']
    options, key = [], None
    for lab, (text, is_c) in zip(labels, entries):
        options.append({'label':lab,'text':text,'is_correct':is_c,
                        'provenance':'generated'})
        if is_c: key = lab
    return options, key

print('Inference functions defined.')

In [ ]:
from datetime import datetime, timezone

# Test set: dùng câu hỏi từ qa_pairs (test split) làm "yêu cầu"
test_df = df[df['split'] == 'test'].copy()
# Lấy tối đa 300 câu để tiết kiệm thời gian (đủ để so sánh với 2 phương pháp kia)
test_df = test_df.sample(n=min(300, len(test_df)), random_state=42).reset_index(drop=True)
print(f'Inference trên {len(test_df)} mẫu test...')

WH_BLOOM_MAP = {
    'thoi_gian': 'nhan_biet',
    'nhan_vat':  'nhan_biet',
    'dia_diem':  'nhan_biet',
    'su_kien':   'thong_hieu',
    'nguyen_nhan': 'thong_hieu',
    'y_nghia':   'van_dung',
}

items = []
skipped = 0

for i, row in test_df.iterrows():
    context  = row['context']
    answer   = row['answer_text']
    wh       = row['wh_type']
    ctx_id   = row['context_id']
    title    = row['title']
    is_vn    = bool(row['is_vietnam'])

    # Stage A: sinh câu hỏi
    question = generate_question(context, answer, wh)
    if not question or not question.strip():
        skipped += 1; continue

    # Stage B: sinh distractor
    distractors = generate_distractors(question, answer, context)
    options, key = finalize_options(answer, distractors, seed=i)
    if options is None:
        skipped += 1; continue

    # Evidence
    ev_sent = next(
        (s.strip() for s in context.split('.')
         if answer in s), context.split('.')[0].strip())
    found = answer in ev_sent

    # Build item (format tương thích schema)
    item_id = 'itm_' + hashlib.md5(f'{ctx_id}|{question}'.encode()).hexdigest()[:12]
    item = {
        'schema_version': '1.0',
        'item_id': item_id,
        'generator': {
            'method': 'vit5_ft',
            'variant': 'vit5_base_stage_ab',
            'model_name': 'VietAI/vit5-base',
        },
        'source': {
            'dataset': 'uit_viquad2',
            'context_id': ctx_id,
            'context': context,
            'title': title,
            'is_vietnam': is_vn,
        },
        'request': {
            'bloom_requested': WH_BLOOM_MAP.get(wh, 'nhan_biet'),
            'wh_type_requested': wh,
        },
        'question': question if question.endswith('?') else question + '?',
        'options': options,
        'answer_key': key,
        'answer_text': answer,
        'evidence': {
            'sentence': ev_sent,
            'found_in_context': found,
        },
        'metadata': {
            'wh_type_detected': wh,
            'bloom_predicted': WH_BLOOM_MAP.get(wh, 'nhan_biet'),
            'topic': title,
            'tags': [wh, 'vn' if is_vn else 'world'],
        },
        'generation_trace': {
            'latency_ms': None,
            'cost_usd': 0.0,
            'n_llm_calls': 0,
        },
        'verification': None,
        'created_at': datetime.now(timezone.utc).isoformat(),
    }
    items.append(item)

    if (i+1) % 50 == 0:
        print(f'  {i+1}/{len(test_df)} done | sinh được {len(items)} | bỏ {skipped}')

print(f'\nHoàn tất: {len(items)} câu hỏi | bỏ {skipped}')

In [ ]:
# Lưu ra Drive
out_path = f'{PROJECT_DIR}/data/generated/vit5_ft.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for item in items:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

from collections import Counter
print(f'\n=== ViT5 GENERATOR ===')
print(f'  Câu hỏi sinh được : {len(items)} / {len(test_df)}')
print(f'  VQR thô           : {len(items)/len(test_df):.0%}')
ev_ok = sum(1 for i in items if i['evidence']['found_in_context'])
print(f'  Evidence in ctx   : {ev_ok}/{len(items)} = {ev_ok/max(len(items),1):.0%}')
print(f'  Chi phí API       : $0.00 (offline)')
by_wh = Counter(i['request']['wh_type_requested'] for i in items)
print('\n  Phân bố wh_type:')
for wh, n in sorted(by_wh.items(), key=lambda x: -x[1]):
    print(f'    {wh:14s}: {n}')
print(f'\nĐã lưu -> {out_path}')

if items:
    it = items[0]
    print('\n[MẪU ĐẦU RA]')
    print(f"  Q: {it['question']}")
    for o in it['options']:
        mark = ' ←' if o['is_correct'] else ''
        print(f"  {o['label']}. {o['text']}{mark}")
    print(f"  Evidence: {it['evidence']['sentence'][:80]}...")

## 12. Tóm tắt đường dẫn file kết quả

Sau khi chạy xong, copy 3 file này về máy local vào `data/generated/`:

```
files_v1/
  models/
    vit5_qg/   ← model Stage A đã train
    vit5_dg/   ← model Stage B đã train
  data/generated/
    rule_based.jsonl   (đã có)
    rag_llm.jsonl      (đã có)
    vit5_ft.jsonl      ← file này
```

Sau đó chạy Verifier trên máy local để so sánh cả 3 phương pháp.